load the FAQ data

In [2]:
from ingest import load_faq_data
documents = load_faq_data()

We'll generate questions only for the LLM Zoomcamp FAQ

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [4]:
documents = documents_llm

In [6]:
documents[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [7]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


- The ID becomes the label in our ground truth dataset. We generate questions from a document, so we know that this document holds the answer. Later, search evaluation checks whether search brings back the document with this ID

## Generating questions with structured output

In [8]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
# The instructions for the LLM:

data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [10]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [11]:
# Prepare the document as JSON

import json

user_prompt = json.dumps(doc)

In [12]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [13]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [14]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [15]:
result = response.output_parsed

print(result)

questions=['I just found this course — can I still sign up and take part now?', 'Am I too late to join the course, or is it still open to new students?', 'If I start the course now, can I still get the certificate somehow?', 'What’s the deadline if I want the course certificate after joining late?', 'Can I join the course after it has already started, and what do I need to do to get certified?']


In [16]:
print(result.questions)

['I just found this course — can I still sign up and take part now?', 'Am I too late to join the course, or is it still open to new students?', 'If I start the course now, can I still get the certificate somehow?', 'What’s the deadline if I want the course certificate after joining late?', 'Can I join the course after it has already started, and what do I need to do to get certified?']


In [17]:
from evaluation_utils import llm_structured

In [18]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Can I still start the course if I just found it now?', 'Is it too late to join the course after it already started?', 'If I join the course late, can I still get a certificate?', "What do I need to do to be eligible for the certificate if I'm joining now?", 'Can I submit the project later and still receive the course certificate?']


### Tracking cost

In [19]:
usage.input_tokens, usage.output_tokens

(207, 86)

In [20]:
from evaluation_utils import calc_price

In [21]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525,
 'output_cost': 0.00038700000000000003,
 'total_cost': 0.00054225}

In [23]:
# convert these questions into ground truth records

records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still start the course if I just found it now?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to join the course after it already started?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course late, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': "What do I need to do to be eligible for the certificate if I'm joining now?",
  'document': '74eb249bbf'},
 {'question': 'Can I submit the project later and still receive the course certificate?',
  'document': '74eb249bbf'}]

In [24]:
import pandas as pd

In [25]:
pd.DataFrame(records)

,question,document
0,Can I still start the course if I just found i...,74eb249bbf
1,Is it too late to join the course after it alr...,74eb249bbf
2,"If I join the course late, can I still get a c...",74eb249bbf
3,What do I need to do to be eligible for the ce...,74eb249bbf
4,Can I submit the project later and still recei...,74eb249bbf
